# FUSE in Feature-Rich Settings: Strategies 1–4

This notebook implements four strategies to extend FUSE (a modularity-based semi-supervised graph embedding) to **feature-rich** settings. Each strategy is self-contained and can be run independently.

| Strategy | Name | Core Idea |
|---|---|---|
| 1 | Feature-Initialized Embeddings | Project node features into embedding space as initialization |
| 2 | Feature-Aware Attention in Random Walks | Blend structural + feature cosine similarity in walk attention |
| 3 | Feature Reconstruction Gradient | Add a gradient term that encourages embeddings to reconstruct node features |
| 4 | Feature-Augmented Adjacency | Build a hybrid adjacency combining structural edges + feature-kNN edges |

Results are saved to `./feature rich fuse/`.

## 0. Imports & Setup

In [3]:
import os, time, random, pickle
import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── PyTorch / PyG (for data loading) ──────────────────────────────────────────
import torch
from torch_geometric.datasets import Planetoid, WikiCS, Amazon
from torch_geometric.data import Data as PyGData
from spektral.datasets import Cora

# ── TensorFlow / Spektral (for classifiers) ────────────────────────────────────
import tensorflow as tf
from spektral.layers import GCNConv, GATConv, GraphSageConv

import logging
logging.getLogger('gensim').setLevel(logging.ERROR)

SAVE_DIR = './feature rich fuse'
os.makedirs(SAVE_DIR, exist_ok=True)
DEFAULT_EMB_DIM = 150
print(f'Results will be saved to: {os.path.abspath(SAVE_DIR)}')

Results will be saved to: C:\Users\user\Downloads\FUSE\modularity_intergrated_version\modularity_intergrated_version\feature rich fuse


## 1. Dataset Loader

In [5]:
def load_dataset(dataset_name, root='.'):
    """
    Returns dict with keys: x, a, y, labels, G, pyg_data
    Supported: cora, citeseer, pubmed, wikics, photo
    """
    name = dataset_name.lower()
    if name == 'cora':
        data = Cora()
        graph = data.graphs[0]
        x = graph.x
        a = graph.a.tocsr() if sp.issparse(graph.a) else csr_matrix(graph.a)
        y_onehot = graph.y
        labels = np.argmax(y_onehot, axis=1)
    elif name in ('citeseer', 'pubmed'):
        data = Planetoid(root=root, name=dataset_name.capitalize())
        d = data[0]
        x = d.x.numpy()
        edge_index = d.edge_index.numpy()
        labels = d.y.numpy()
        num_nodes = x.shape[0]
        a = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            s, t = edge_index[:, i]
            a[s, t] = 1; a[t, s] = 1
        a = a.tocsr()
        y_onehot = np.eye(labels.max() + 1)[labels]
    elif name == 'wikics':
        data = WikiCS(root=root)
        d = data[0]
        x = d.x.numpy()
        edge_index = d.edge_index.numpy()
        labels = d.y.numpy()
        num_nodes = x.shape[0]
        a = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            s, t = edge_index[:, i]
            a[s, t] = 1; a[t, s] = 1
        a = a.tocsr()
        y_onehot = np.eye(labels.max() + 1)[labels]
    elif name in ('photo', 'amazon-photo', 'amazon_photos'):
        data = Amazon(root=root, name='photo')
        d = data[0]
        x = d.x.numpy()
        edge_index = d.edge_index.numpy()
        labels = d.y.numpy()
        num_nodes = x.shape[0]
        a = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            s, t = edge_index[:, i]
            a[s, t] = 1; a[t, s] = 1
        a = a.tocsr()
        y_onehot = np.eye(labels.max() + 1)[labels]
    else:
        raise ValueError(f'Unknown dataset: {dataset_name}')

    row, col = a.nonzero()
    pyg = PyGData(
        x=torch.tensor(x, dtype=torch.float),
        edge_index=torch.tensor(np.vstack([row, col]), dtype=torch.long),
        y=torch.tensor(labels, dtype=torch.long)
    )
    G = nx.from_scipy_sparse_array(a)
    return dict(x=np.array(x, dtype=float), a=a.tocsr(),
                y=np.array(y_onehot, dtype=float),
                labels=np.array(labels, dtype=int), G=G, pyg_data=pyg)


def create_label_mask(labels, mask_frac=0.7, seed=None):
    """mask_frac = fraction of nodes whose labels are KNOWN (kept)."""
    n = len(labels)
    rng = np.random.RandomState(seed) if seed is not None else np.random
    k = int(round(n * (1 - mask_frac)))
    hidden = rng.choice(np.arange(n), size=k, replace=False)
    masked = np.array(labels, dtype=int)
    masked[hidden] = -1
    label_mask = masked != -1
    return masked, label_mask, hidden


print('Dataset loader defined.')

Dataset loader defined.


## 2. Shared FUSE Helpers

In [7]:
def perform_labeled_random_walks(G, label_mask, labels,
                                  num_walks=10, walk_length=5, walk_length_labelled=3):
    walks = {node: [] for node in G.nodes()}
    for node in G.nodes():
        for _ in range(num_walks):
            walk = [node]
            labeled_count = 0
            for _ in range(walk_length - 1):
                cur = walk[-1]
                nbrs = list(G.neighbors(cur))
                if not nbrs:
                    break
                labeled_nbrs = [nb for nb in nbrs if label_mask[nb]]
                if labeled_nbrs and labeled_count < walk_length_labelled:
                    nxt = random.choice(labeled_nbrs)
                    labeled_count += 1
                else:
                    nxt = random.choice(nbrs)
                walk.append(nxt)
            walks[node].extend([nb for nb in walk if label_mask[nb]])
    return walks


def compute_attention_weights_base(S, labeled_nodes):
    """Original structural-only attention."""
    weights = {}
    for node, labeled in labeled_nodes.items():
        if labeled:
            sims = {nb: float(np.dot(S[node], S[nb])) for nb in labeled}
            exp_s = {nb: np.exp(v) for nb, v in sims.items()}
            tot = sum(exp_s.values()) or 1.0
            weights[node] = {nb: exp_s[nb] / tot for nb in labeled}
    return weights


def modularity_grad(A, S, degrees, m):
    """Compute the modularity gradient for S."""
    neighbor_agg = A @ S
    global_corr = (degrees[:, None] / (2 * m)) * S.sum(axis=0)
    return (1 / (2 * m)) * (neighbor_agg - global_corr)


def supervised_grad(S, labels, label_mask):
    """Pull same-class embeddings towards their class mean."""
    grad = np.zeros_like(S)
    if not np.any(label_mask):
        return grad
    for lab in np.unique(labels[label_mask]):
        mask = (labels == lab) & label_mask
        if mask.sum() == 0:
            continue
        mean_emb = np.mean(S[mask], axis=0, keepdims=True)
        grad[mask] = S[mask] - mean_emb
    return grad


def semi_supervised_grad(S, n, label_mask, attention_weights):
    """Pull unlabeled nodes towards their attention-weighted labeled neighbors."""
    grad = np.zeros_like(S)
    for i in range(n):
        if (not label_mask[i]) and (i in attention_weights):
            w_emb = sum(w * S[j] for j, w in attention_weights[i].items())
            grad[i] = S[i] - w_emb
    return grad


print('Shared FUSE helpers defined.')

Shared FUSE helpers defined.


## 3. Classifier Infrastructure (Spektral / TF)

In [9]:
class NoMaskGCNConv(GCNConv):
    def compute_mask(self, inputs, mask=None): return None
    def call(self, inputs, training=None, mask=None): return super().call(inputs, mask=None)

class NoMaskGATConv(GATConv):
    def compute_mask(self, inputs, mask=None): return None
    def call(self, inputs, training=None, mask=None): return super().call(inputs, mask=None)

class GCN(tf.keras.Model):
    def __init__(self, n_labels, seed=42):
        super().__init__()
        init = tf.keras.initializers.GlorotUniform(seed=seed)
        self.conv1 = NoMaskGCNConv(16, activation='relu', kernel_initializer=init)
        self.conv2 = NoMaskGCNConv(n_labels, activation='softmax', kernel_initializer=init)
    def call(self, inputs, training=False):
        x, a = inputs
        h = self.conv1([x, a])
        return self.conv2([h, a]), h

class GAT(tf.keras.Model):
    def __init__(self, n_labels, seed=42):
        super().__init__()
        init = tf.keras.initializers.GlorotUniform(seed=seed)
        self.conv1 = NoMaskGATConv(16, attn_heads=8, concat_heads=True, activation='elu', kernel_initializer=init)
        self.conv2 = NoMaskGATConv(n_labels, attn_heads=1, concat_heads=False, activation='softmax', kernel_initializer=init)
    def call(self, inputs, training=False):
        x, a = inputs
        h = self.conv1([x, a])
        return self.conv2([h, a]), h

class GraphSAGE(tf.keras.Model):
    def __init__(self, n_labels, seed=42):
        super().__init__()
        init = tf.keras.initializers.GlorotUniform(seed=seed)
        self.conv1 = GraphSageConv(16, activation='relu', kernel_initializer=init)
        self.conv2 = GraphSageConv(n_labels, activation='softmax', kernel_initializer=init)
    def call(self, inputs, training=False):
        x, a = inputs
        h = self.conv1([x, a])
        return self.conv2([h, a]), h


def sparse_to_tf(A):
    A = A.tocoo()
    indices = np.column_stack((A.row, A.col))
    return tf.sparse.SparseTensor(indices=indices, values=A.data, dense_shape=A.shape)


def train_and_eval(E, A, y_onehot, labels_int, label_mask,
                   clf_name='gcn', epochs=200, seed=42):
    """Train classifier on embedding E; evaluate on unlabeled nodes."""
    tf.random.set_seed(seed)
    np.random.seed(seed)

    num_classes = y_onehot.shape[1]
    train_idx = np.where(label_mask)[0]
    X_tr = tf.convert_to_tensor(E[train_idx].astype(np.float32))
    y_tr = y_onehot[train_idx]
    A_tr = sparse_to_tf(A[train_idx, :][:, train_idx])

    clf_map = {'gcn': GCN, 'gat': GAT, 'graphsage': GraphSAGE}
    model = clf_map[clf_name.lower()](num_classes, seed=seed)
    opt = tf.keras.optimizers.Adam(1e-2)
    loss_fn = tf.keras.losses.CategoricalCrossentropy()

    t0 = time.time()
    for _ in range(epochs):
        with tf.GradientTape() as tape:
            preds, _ = model([X_tr, A_tr], training=True)
            loss = loss_fn(y_tr, preds)
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
    train_time = time.time() - t0

    X_full = tf.convert_to_tensor(E.astype(np.float32))
    A_full = sparse_to_tf(A)
    preds_all, _ = model([X_full, A_full], training=False)
    pred_int = tf.argmax(preds_all, axis=1).numpy()

    masked_idx = np.where(~label_mask)[0]
    acc = accuracy_score(labels_int[masked_idx], pred_int[masked_idx])
    f1  = f1_score(labels_int[masked_idx], pred_int[masked_idx], average='macro')
    return {'accuracy': float(acc), 'f1_score': float(f1),
            'train_time': float(train_time), 'classifier': clf_name}


print('Classifier infrastructure defined.')

Classifier infrastructure defined.


---
## Strategy 1 — Feature-Initialized Embeddings

**Idea:** Project node features `X` into the `k`-dimensional embedding space using Truncated SVD, then use that projection as the starting point for FUSE's orthonormalization and gradient ascent. Features shape the *initial geometry* of the embedding; modularity + supervision steer it from there.

**When it helps:** When feature clusters roughly align with community structure; a better starting point means fewer iterations to converge.

**Limitation:** Features only affect initialization, not the optimization trajectory. If features are noisy or misaligned with topology, the benefit is limited.

In [11]:
def fuse_strategy1_feature_init(
        G, labels, label_mask, X,
        k=DEFAULT_EMB_DIM, eta=0.01,
        lambda_supervised=1.0, lambda_semi=2.0,
        iterations=200, seed=None,
        num_walks=10, walk_length=5, walk_length_labelled=3):
    """
    FUSE Strategy 1: Feature-Initialized Embeddings.

    Node features X (n x d) are projected to k dimensions via TruncatedSVD
    and used as the initial embedding S_0 instead of random Gaussian noise.
    The rest of the FUSE optimization is unchanged.

    Parameters
    ----------
    G            : networkx Graph
    labels       : int array (n,)  — full integer labels
    label_mask   : bool array (n,) — True = label is known
    X            : float array (n, d) — node feature matrix
    k            : embedding dimension
    eta          : gradient step size
    lambda_supervised  : weight for supervised gradient
    lambda_semi        : weight for semi-supervised gradient
    iterations         : number of gradient steps
    seed               : random seed
    num_walks, walk_length, walk_length_labelled : random-walk hyper-parameters

    Returns
    -------
    S : (n, k) numpy array — learned embedding
    """
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    A = csr_matrix(nx.to_scipy_sparse_array(G, format='csr'))
    degrees = np.array(A.sum(axis=1)).flatten()
    m = max(G.number_of_edges(), 1)
    n = A.shape[0]

    # ── Feature initialization ───────────────────────────────────────────────
    d = X.shape[1]
    if d >= k:
        svd = TruncatedSVD(n_components=k, random_state=seed)
        S = svd.fit_transform(X)                   # (n, k)
    else:
        # Pad with zeros if feature dim < k
        S = np.zeros((n, k))
        S[:, :d] = X

    S, _ = np.linalg.qr(S)                         # Orthonormalize
    # ─────────────────────────────────────────────────────────────────────────

    labeled_walks = perform_labeled_random_walks(
        G, label_mask, labels, num_walks, walk_length, walk_length_labelled)
    attention_weights = compute_attention_weights_base(S, labeled_walks)

    for _ in range(iterations):
        g_mod  = modularity_grad(A, S, degrees, m)
        g_sup  = supervised_grad(S, labels, label_mask)
        g_semi = semi_supervised_grad(S, n, label_mask, attention_weights)
        S += eta * (g_mod - lambda_supervised * g_sup - lambda_semi * g_semi)
        S, _ = np.linalg.qr(S)

    return S

### Strategy 1 — Demo Run

In [13]:
DEMO_DATASET = 'cora'
DEMO_SEED    = 42
DEMO_MF      = 0.7   # fraction of labels KNOWN
CLASSIFIERS  = ['gcn', 'gat', 'graphsage']

ds = load_dataset(DEMO_DATASET)
_, label_mask, _ = create_label_mask(ds['labels'], mask_frac=DEMO_MF, seed=DEMO_SEED)

t0 = time.time()
E1 = fuse_strategy1_feature_init(
    ds['G'], ds['labels'], label_mask, ds['x'],
    k=DEFAULT_EMB_DIM, seed=DEMO_SEED)
emb_time = time.time() - t0
print(f'Strategy 1 embedding shape: {E1.shape}  |  time: {emb_time:.1f}s')

s1_results = []
for clf in CLASSIFIERS:
    r = train_and_eval(E1, ds['a'], ds['y'], ds['labels'], label_mask, clf_name=clf, seed=DEMO_SEED)
    r.update({'dataset': DEMO_DATASET, 'seed': DEMO_SEED, 'mask_frac': DEMO_MF,
              'strategy': 'strategy1_feature_init', 'emb_time': emb_time})
    s1_results.append(r)
    print(f'  [{clf}]  acc={r["accuracy"]:.4f}  f1={r["f1_score"]:.4f}')

df_s1 = pd.DataFrame(s1_results)
out_path = os.path.join(SAVE_DIR, 'strategy1_feature_init_results.csv')
df_s1.to_csv(out_path, index=False)
print(f'\nResults saved to: {out_path}')

Strategy 1 embedding shape: (2708, 150)  |  time: 15.2s
  [gcn]  acc=0.8017  f1=0.7898
  [gat]  acc=0.8633  f1=0.8523
  [graphsage]  acc=0.8571  f1=0.8391

Results saved to: ./feature rich fuse\strategy1_feature_init_results.csv


---
## Strategy 2 — Feature-Aware Attention in Random Walks

**Idea:** In the semi-supervised gradient, unlabeled nodes are pulled toward labeled neighbors weighted by attention. Originally, attention is computed purely from structural embedding dot-products. Here we blend in **feature cosine similarity** so that labeled nodes that are *both* structurally and featurally close to an unlabeled node exert stronger influence.

$$\text{sim}(u, v) = \alpha \cdot \langle S_u, S_v \rangle + (1-\alpha) \cdot \cos(X_u, X_v)$$

**When it helps:** When features carry complementary information to topology — e.g., two nodes connected in the graph but in different feature clusters should not dominate each other's attention.

In [15]:
def compute_attention_weights_feature_aware(S, labeled_nodes, X, alpha=0.5):
    """
    Blended attention: alpha * structural_sim + (1-alpha) * feature_cosine_sim.

    Parameters
    ----------
    S            : current embedding (n, k)
    labeled_nodes: dict {node -> list-of-labeled-neighbors} from random walks
    X            : node features (n, d)
    alpha        : weight on structural similarity (0 = pure feature, 1 = pure structural)
    """
    X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    weights = {}
    for node, labeled in labeled_nodes.items():
        if labeled:
            sims = {}
            for nb in labeled:
                struct_sim = float(np.dot(S[node], S[nb]))
                feat_sim   = float(np.dot(X_norm[node], X_norm[nb]))
                sims[nb]   = alpha * struct_sim + (1.0 - alpha) * feat_sim
            exp_s = {nb: np.exp(v) for nb, v in sims.items()}
            tot   = sum(exp_s.values()) or 1.0
            weights[node] = {nb: exp_s[nb] / tot for nb in labeled}
    return weights


def fuse_strategy2_feature_attention(
        G, labels, label_mask, X,
        k=DEFAULT_EMB_DIM, eta=0.01,
        lambda_supervised=1.0, lambda_semi=2.0,
        iterations=200, alpha=0.5, seed=None,
        num_walks=10, walk_length=5, walk_length_labelled=3):
    """
    FUSE Strategy 2: Feature-Aware Attention in Random Walks.

    The attention weights used in the semi-supervised gradient are computed
    by blending structural embedding similarity with feature cosine similarity,
    controlled by the `alpha` parameter.

    Parameters
    ----------
    alpha : float in [0, 1]
        Weight on structural similarity. alpha=1.0 recovers the original FUSE;
        alpha=0.0 uses only feature cosine similarity.

    All other parameters same as Strategy 1.

    Returns
    -------
    S : (n, k) numpy array
    """
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    A = csr_matrix(nx.to_scipy_sparse_array(G, format='csr'))
    degrees = np.array(A.sum(axis=1)).flatten()
    m = max(G.number_of_edges(), 1)
    n = A.shape[0]

    S = np.random.randn(n, k)
    S, _ = np.linalg.qr(S)

    labeled_walks = perform_labeled_random_walks(
        G, label_mask, labels, num_walks, walk_length, walk_length_labelled)

    # ── Feature-aware attention ──────────────────────────────────────────────
    attention_weights = compute_attention_weights_feature_aware(
        S, labeled_walks, X, alpha=alpha)
    # ─────────────────────────────────────────────────────────────────────────

    for _ in range(iterations):
        g_mod  = modularity_grad(A, S, degrees, m)
        g_sup  = supervised_grad(S, labels, label_mask)
        g_semi = semi_supervised_grad(S, n, label_mask, attention_weights)
        S += eta * (g_mod - lambda_supervised * g_sup - lambda_semi * g_semi)
        S, _ = np.linalg.qr(S)

    return S

### Strategy 2 — Demo Run

In [17]:
t0 = time.time()
E2 = fuse_strategy2_feature_attention(
    ds['G'], ds['labels'], label_mask, ds['x'],
    k=DEFAULT_EMB_DIM, alpha=0.5, seed=DEMO_SEED)
emb_time2 = time.time() - t0
print(f'Strategy 2 embedding shape: {E2.shape}  |  time: {emb_time2:.1f}s')

s2_results = []
for clf in CLASSIFIERS:
    r = train_and_eval(E2, ds['a'], ds['y'], ds['labels'], label_mask, clf_name=clf, seed=DEMO_SEED)
    r.update({'dataset': DEMO_DATASET, 'seed': DEMO_SEED, 'mask_frac': DEMO_MF,
              'strategy': 'strategy2_feature_attention', 'alpha': 0.5, 'emb_time': emb_time2})
    s2_results.append(r)
    print(f'  [{clf}]  acc={r["accuracy"]:.4f}  f1={r["f1_score"]:.4f}')

df_s2 = pd.DataFrame(s2_results)
out_path = os.path.join(SAVE_DIR, 'strategy2_feature_attention_results.csv')
df_s2.to_csv(out_path, index=False)
print(f'\nResults saved to: {out_path}')

Strategy 2 embedding shape: (2708, 150)  |  time: 14.9s
  [gcn]  acc=0.8214  f1=0.8031
  [gat]  acc=0.8571  f1=0.8440
  [graphsage]  acc=0.8411  f1=0.8186

Results saved to: ./feature rich fuse\strategy2_feature_attention_results.csv


---
## Strategy 3 — Feature Reconstruction Gradient

**Idea:** Add a fourth gradient term to the FUSE objective that penalises the inability to reconstruct node features from the embedding:

$$\mathcal{L}_{\text{feat}} = \|SW - X\|_F^2, \quad W = S^{+}X \text{ (least-squares decoder)}$$

The gradient with respect to `S` is:
$$\nabla_S \mathcal{L}_{\text{feat}} = (SW - X) W^\top$$

This is the most principled approach: FUSE now jointly optimizes for modularity, label consistency, semi-supervised propagation, **and** feature reconstruction — directly analogous to how VGAE reconstructs adjacency.

**When it helps:** When features carry information that structural modularity alone would miss (e.g., heterophilic graphs, or rich bag-of-words features).

In [19]:
def fuse_strategy3_feature_reconstruction(
        G, labels, label_mask, X,
        k=DEFAULT_EMB_DIM, eta=0.01,
        lambda_supervised=1.0, lambda_semi=2.0, lambda_feature=0.5,
        iterations=200, seed=None,
        num_walks=10, walk_length=5, walk_length_labelled=3):
    """
    FUSE Strategy 3: Feature Reconstruction Gradient.

    Adds an explicit gradient term encouraging the embedding S to be linearly
    decodable back to node features X. A least-squares decoder W = S^+ X is
    computed at each step, and the gradient -lambda_feature * (SW - X) W^T
    is added to the update.

    Parameters
    ----------
    lambda_feature : float
        Weight for the feature reconstruction gradient term.
        Larger values force the embedding closer to the feature space;
        smaller values let modularity dominate.

    All other parameters same as Strategy 1.

    Returns
    -------
    S : (n, k) numpy array
    """
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    A = csr_matrix(nx.to_scipy_sparse_array(G, format='csr'))
    degrees = np.array(A.sum(axis=1)).flatten()
    m = max(G.number_of_edges(), 1)
    n = A.shape[0]

    S = np.random.randn(n, k)
    S, _ = np.linalg.qr(S)

    labeled_walks = perform_labeled_random_walks(
        G, label_mask, labels, num_walks, walk_length, walk_length_labelled)
    attention_weights = compute_attention_weights_base(S, labeled_walks)

    X_arr = np.array(X, dtype=float)

    for _ in range(iterations):
        g_mod  = modularity_grad(A, S, degrees, m)
        g_sup  = supervised_grad(S, labels, label_mask)
        g_semi = semi_supervised_grad(S, n, label_mask, attention_weights)

        # ── Feature reconstruction gradient ──────────────────────────────────
        # Least-squares decoder: W = argmin ||SW - X||^2  →  W = pinv(S) @ X
        W, _, _, _ = np.linalg.lstsq(S, X_arr, rcond=None)  # (k, d)
        residual    = S @ W - X_arr                           # (n, d)
        g_feat      = residual @ W.T                          # (n, k)  gradient of ||SW-X||^2 w.r.t. S
        # ─────────────────────────────────────────────────────────────────────

        S += eta * (g_mod
                    - lambda_supervised * g_sup
                    - lambda_semi       * g_semi
                    - lambda_feature    * g_feat)   # subtract: we minimize feature loss
        S, _ = np.linalg.qr(S)

    return S

### Strategy 3 — Demo Run

In [21]:
t0 = time.time()
E3 = fuse_strategy3_feature_reconstruction(
    ds['G'], ds['labels'], label_mask, ds['x'],
    k=DEFAULT_EMB_DIM, lambda_feature=0.5, seed=DEMO_SEED)
emb_time3 = time.time() - t0
print(f'Strategy 3 embedding shape: {E3.shape}  |  time: {emb_time3:.1f}s')

s3_results = []
for clf in CLASSIFIERS:
    r = train_and_eval(E3, ds['a'], ds['y'], ds['labels'], label_mask, clf_name=clf, seed=DEMO_SEED)
    r.update({'dataset': DEMO_DATASET, 'seed': DEMO_SEED, 'mask_frac': DEMO_MF,
              'strategy': 'strategy3_feature_reconstruction', 'lambda_feature': 0.5,
              'emb_time': emb_time3})
    s3_results.append(r)
    print(f'  [{clf}]  acc={r["accuracy"]:.4f}  f1={r["f1_score"]:.4f}')

df_s3 = pd.DataFrame(s3_results)
out_path = os.path.join(SAVE_DIR, 'strategy3_feature_reconstruction_results.csv')
df_s3.to_csv(out_path, index=False)
print(f'\nResults saved to: {out_path}')

Strategy 3 embedding shape: (2708, 150)  |  time: 39.2s
  [gcn]  acc=0.7894  f1=0.7789
  [gat]  acc=0.8596  f1=0.8517
  [graphsage]  acc=0.7956  f1=0.7676

Results saved to: ./feature rich fuse\strategy3_feature_reconstruction_results.csv


---
## Strategy 4 — Feature-Augmented Adjacency

**Idea:** Construct a *hybrid adjacency* before running FUSE, blending the structural adjacency with a feature-kNN graph:

$$A_{\text{hybrid}} = (1 - \beta)\,A_{\text{structural}} + \beta\,A_{\text{feature-kNN}}$$

where $A_{\text{feature-kNN}}$ connects each node to its `knn_k` nearest neighbors in feature space (by cosine similarity). The FUSE core algorithm is then run **unchanged** on $A_{\text{hybrid}}$.

**When it helps:** When the feature space reveals communities not captured by the graph edges alone (e.g., co-citation graphs where nodes cite the same papers but are not directly linked).

**Advantage:** Zero changes to the core FUSE algorithm — fully backward compatible.

In [23]:
def build_feature_augmented_adjacency(A, X, beta=0.3, knn_k=10):
    """
    Build a hybrid adjacency: (1-beta)*A_structural + beta*A_feature_kNN.

    Parameters
    ----------
    A     : (n, n) scipy sparse CSR — original structural adjacency
    X     : (n, d) numpy array — node features
    beta  : float in [0, 1] — weight on the feature-kNN adjacency
    knn_k : int — number of nearest feature neighbors per node

    Returns
    -------
    A_hybrid : scipy sparse CSR matrix
    """
    n = X.shape[0]
    feat_sim = cosine_similarity(X)                              # (n, n)
    np.fill_diagonal(feat_sim, 0)                                # no self-loops

    # Keep top-knn_k neighbors per row
    thresh = np.partition(feat_sim, -knn_k, axis=1)[:, -knn_k]
    feat_adj = (feat_sim >= thresh[:, None]).astype(np.float32)
    np.fill_diagonal(feat_adj, 0)

    feat_adj_sp = csr_matrix(feat_adj)
    A_hybrid = (1.0 - beta) * A + beta * feat_adj_sp
    return A_hybrid


def fuse_strategy4_augmented_adjacency(
        G, labels, label_mask, X,
        k=DEFAULT_EMB_DIM, eta=0.01,
        lambda_supervised=1.0, lambda_semi=2.0,
        iterations=200, beta=0.3, knn_k=10, seed=None,
        num_walks=10, walk_length=5, walk_length_labelled=3):
    """
    FUSE Strategy 4: Feature-Augmented Adjacency.

    A feature-kNN graph is constructed from node features and blended with the
    structural adjacency. The resulting hybrid graph is then used as input to
    the standard FUSE algorithm with no further changes.

    Parameters
    ----------
    beta  : float in [0, 1] — weight given to feature-kNN edges.
            beta=0.0 recovers the original FUSE; beta=1.0 is pure feature-kNN.
    knn_k : number of feature nearest-neighbors per node.

    All other parameters same as Strategy 1.

    Returns
    -------
    S : (n, k) numpy array
    """
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    A_struct = csr_matrix(nx.to_scipy_sparse_array(G, format='csr'))

    # ── Feature-augmented adjacency ──────────────────────────────────────────
    A = build_feature_augmented_adjacency(A_struct, np.array(X, dtype=float),
                                           beta=beta, knn_k=knn_k)
    G_hybrid = nx.from_scipy_sparse_array(A)           # rebuild graph for walks
    # ─────────────────────────────────────────────────────────────────────────

    degrees = np.array(A.sum(axis=1)).flatten()
    m = max(A.nnz // 2, 1)
    n = A.shape[0]

    S = np.random.randn(n, k)
    S, _ = np.linalg.qr(S)

    labeled_walks = perform_labeled_random_walks(
        G_hybrid, label_mask, labels, num_walks, walk_length, walk_length_labelled)
    attention_weights = compute_attention_weights_base(S, labeled_walks)

    for _ in range(iterations):
        g_mod  = modularity_grad(A, S, degrees, m)
        g_sup  = supervised_grad(S, labels, label_mask)
        g_semi = semi_supervised_grad(S, n, label_mask, attention_weights)
        S += eta * (g_mod - lambda_supervised * g_sup - lambda_semi * g_semi)
        S, _ = np.linalg.qr(S)

    return S

### Strategy 4 — Demo Run

In [25]:
t0 = time.time()
E4 = fuse_strategy4_augmented_adjacency(
    ds['G'], ds['labels'], label_mask, ds['x'],
    k=DEFAULT_EMB_DIM, beta=0.3, knn_k=10, seed=DEMO_SEED)
emb_time4 = time.time() - t0
print(f'Strategy 4 embedding shape: {E4.shape}  |  time: {emb_time4:.1f}s')

s4_results = []
for clf in CLASSIFIERS:
    r = train_and_eval(E4, ds['a'], ds['y'], ds['labels'], label_mask, clf_name=clf, seed=DEMO_SEED)
    r.update({'dataset': DEMO_DATASET, 'seed': DEMO_SEED, 'mask_frac': DEMO_MF,
              'strategy': 'strategy4_augmented_adj', 'beta': 0.3, 'knn_k': 10,
              'emb_time': emb_time4})
    s4_results.append(r)
    print(f'  [{clf}]  acc={r["accuracy"]:.4f}  f1={r["f1_score"]:.4f}')

df_s4 = pd.DataFrame(s4_results)
out_path = os.path.join(SAVE_DIR, 'strategy4_augmented_adj_results.csv')
df_s4.to_csv(out_path, index=False)
print(f'\nResults saved to: {out_path}')

Strategy 4 embedding shape: (2708, 150)  |  time: 23.3s
  [gcn]  acc=0.8214  f1=0.7948
  [gat]  acc=0.8732  f1=0.8598
  [graphsage]  acc=0.8214  f1=0.7970

Results saved to: ./feature rich fuse\strategy4_augmented_adj_results.csv


---
## Summary — All Strategies on Demo Dataset

In [27]:
all_demo = pd.concat([df_s1, df_s2, df_s3, df_s4], ignore_index=True)
summary = all_demo.groupby(['strategy', 'classifier'])[['accuracy', 'f1_score']].mean().round(4)
print(summary.to_string())

all_demo.to_csv(os.path.join(SAVE_DIR, 'all_strategies_demo_results.csv'), index=False)
print(f'\nCombined results saved to: {os.path.join(SAVE_DIR, "all_strategies_demo_results.csv")}')

                                             accuracy  f1_score
strategy                         classifier                    
strategy1_feature_init           gat           0.8633    0.8523
                                 gcn           0.8017    0.7898
                                 graphsage     0.8571    0.8391
strategy2_feature_attention      gat           0.8571    0.8440
                                 gcn           0.8214    0.8031
                                 graphsage     0.8411    0.8186
strategy3_feature_reconstruction gat           0.8596    0.8517
                                 gcn           0.7894    0.7789
                                 graphsage     0.7956    0.7676
strategy4_augmented_adj          gat           0.8732    0.8598
                                 gcn           0.8214    0.7948
                                 graphsage     0.8214    0.7970

Combined results saved to: ./feature rich fuse\all_strategies_demo_results.csv
